In [29]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("datasets/housing_price_data.csv")

df = pd.get_dummies(df, columns=['City'], drop_first=True)
label_encoder = LabelEncoder()
df['Renovation Status'] = label_encoder.fit_transform(df['Renovation Status'].astype(str))

X = df.drop(columns=["House ID", "Price ($)"])
y = df["Price ($)"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_model = LinearRegression()
grad_model = GradientBoostingRegressor()
rfreg_model = RandomForestRegressor(random_state=42)

reg_model.fit(X_train, y_train)
grad_model.fit(X_train, y_train)
rfreg_model.fit(X_train, y_train)

reg_pred = reg_model.predict(X_test)
grad_pred = grad_model.predict(X_test)
rf_pred = rfreg_model.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Results:")
    print(f"  Mean Absolute Error (MAE): {mae:.2f}")
    print(f"  R² Score: {r2:.2f}")
    print("-" * 40)

evaluate_model("Linear Regression", y_test, reg_pred)
evaluate_model("Gradient Boost", y_test, grad_pred)
evaluate_model("Random Forest", y_test, rf_pred)


Linear Regression Results:
  Mean Absolute Error (MAE): 118060.63
  R² Score: 0.52
----------------------------------------
Gradient Boost Results:
  Mean Absolute Error (MAE): 115952.24
  R² Score: 0.49
----------------------------------------
Random Forest Results:
  Mean Absolute Error (MAE): 116341.43
  R² Score: 0.47
----------------------------------------


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Load and preprocess data
df = pd.read_csv("datasets/housing_price_data.csv")
df = pd.get_dummies(df, columns=['City'], drop_first=True)

label_encoder = LabelEncoder()
df['Renovation Status'] = label_encoder.fit_transform(df['Renovation Status'].astype(str))

X = df.drop(columns=["House ID", "Price ($)"])
y = df["Price ($)"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models and parameter grids
models = {
    "GradientBoosting": {
        "model": GradientBoostingRegressor(),
        "params": {
            "learning_rate": [0.01, 0.05, 0.1],
            "n_estimators": [10],  # 10 epochs/iterations limit
            "max_depth": [2, 3, 4]
        }
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators": [10],  # batch-driven small training
            "max_depth": [4, 6, 8]
        }
    },
    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}  # No hyperparameters to tune
    }
}

# Function to evaluate models
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Results:")
    print(f"  Mean Absolute Error (MAE): {mae:.2f}")
    print(f"  R² Score: {r2:.2f}")
    print("-" * 40)

# Batch-driven grid search
for name, mp in models.items():
    print(f"Running Grid Search for {name}...")
    grid = GridSearchCV(mp["model"], mp["params"], cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    print(f"Best parameters for {name}: {grid.best_params_}")
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    evaluate_model(name, y_test, y_pred)


In [30]:
import matplotlib.pyplot as plt

results_df = pd.DataFrame(grid.cv_results_)
plt.plot(results_df["mean_test_score"], marker='o')
plt.title(f'{name} - Batch Grid Search Performance (10 Epochs)')
plt.xlabel('Parameter Combination Index')
plt.ylabel('Mean R² Score')
plt.grid(True)
plt.show()


NameError: name 'grid' is not defined